# Vérification du WebSocket EMSC

Ce notebook teste séparément l’ouverture de la connexion et la réception d’événements sur le flux public EMSC. L’attente est limitée dans le temps; aucune connexion Redis n’est nécessaire.

## 1. Importer les dépendances et définir l’URL EMSC

Le kernel doit disposer de `websockets` (version 17.1 ou supérieure, présente dans les dépendances de l’API backend).

In [1]:
import asyncio
import json
from websockets.asyncio.client import connect

EMSC_WEBSOCKET: str = "wss://www.seismicportal.eu/standing_order/websocket"
OPEN_TIMEOUT_SECONDS: float = 10
PING_INTERVAL_SECONDS: float = 20
PING_TIMEOUT_SECONDS: float = 20
WAIT_SECONDS: float = 30
MAX_MESSAGES: int = 5

print(f"WebSocket EMSC : {EMSC_WEBSOCKET}")
print(f"Délai d’ouverture : {OPEN_TIMEOUT_SECONDS} s")
print(f"Fenêtre de réception : {WAIT_SECONDS} s, maximum {MAX_MESSAGES} message(s)")

WebSocket EMSC : wss://www.seismicportal.eu/standing_order/websocket
Délai d’ouverture : 10 s
Fenêtre de réception : 30 s, maximum 5 message(s)


## 2. Ouvrir une connexion WebSocket avec délai maximal

L’ouverture est limitée à 10 secondes. Les pings vérifient ensuite que la connexion reste active.

In [2]:
websocket = None
connection_error: str | None = None

try:
    websocket = await connect(
        EMSC_WEBSOCKET,
        open_timeout=OPEN_TIMEOUT_SECONDS,
        ping_interval=PING_INTERVAL_SECONDS,
        ping_timeout=PING_TIMEOUT_SECONDS,
    )
except Exception as error:
    connection_error = f"{type(error).__name__}: {error}"
    print(f"ÉCHEC de connexion : {connection_error}")
else:
    print("CONNEXION ÉTABLIE : handshake WebSocket réussi.")

CONNEXION ÉTABLIE : handshake WebSocket réussi.


## 3. Recevoir et décoder des messages

Le notebook attend jusqu’à 30 secondes et s’arrête après cinq trames. Chaque trame est décodée avec `json.loads`; les erreurs sont comptabilisées sans masquer le résultat du handshake.

In [3]:
raw_messages: list[str | bytes] = []
decoded_messages: list[object] = []
decode_errors: list[str] = []
receive_error: str | None = None

if websocket is None:
    print("Réception ignorée : aucune connexion n’a été établie.")
else:
    deadline = asyncio.get_running_loop().time() + WAIT_SECONDS
    while len(raw_messages) < MAX_MESSAGES:
        remaining_seconds = deadline - asyncio.get_running_loop().time()
        if remaining_seconds <= 0:
            break
        try:
            raw_message = await asyncio.wait_for(
                websocket.recv(), timeout=remaining_seconds
            )
        except asyncio.TimeoutError:
            break
        except Exception as error:
            receive_error = f"{type(error).__name__}: {error}"
            print(f"ERREUR de réception : {receive_error}")
            break

        raw_messages.append(raw_message)
        try:
            decoded_messages.append(json.loads(raw_message))
        except (json.JSONDecodeError, UnicodeDecodeError, TypeError) as error:
            decode_errors.append(f"{type(error).__name__}: {error}")
            print(f"Message {len(raw_messages)} : JSON invalide ({type(error).__name__}).")
        else:
            print(f"Message {len(raw_messages)} : JSON décodé.")

    if not raw_messages and receive_error is None:
        print(f"Aucun message reçu pendant {WAIT_SECONDS} secondes.")

Message 1 : JSON décodé.


## 4. Vérifier la structure des événements JSON

Un événement est considéré valide si le JSON racine est un dictionnaire avec `action` et `data`, et si `data` est lui-même un dictionnaire. Seules quelques clés représentatives sont affichées.

In [4]:
valid_events: list[dict[str, object]] = []
structure_errors: list[str] = []

for message_number, event in enumerate(decoded_messages, start=1):
    if not isinstance(event, dict):
        structure_errors.append(f"message {message_number} : l’objet racine n’est pas un dictionnaire")
        continue

    missing_keys = {"action", "data"} - event.keys()
    if missing_keys:
        structure_errors.append(
            f"message {message_number} : clés absentes {sorted(missing_keys)}"
        )
        continue

    data = event["data"]
    if not isinstance(data, dict):
        structure_errors.append(f"message {message_number} : `data` n’est pas un dictionnaire")
        continue

    valid_events.append(event)
    event_id = data.get("unid", data.get("id", "non fourni"))
    print(
        f"Événement {message_number} valide : action={event['action']!r}, "
        f"id={event_id!r}, clés data={list(data)[:8]}"
    )

for structure_error in structure_errors:
    print(f"Structure invalide : {structure_error}")

Événement 1 valide : action='update', id='20260928_0000183', clés data=['type', 'geometry', 'id', 'properties']


## 5. Afficher un bilan des tests et fermer la connexion

Le handshake réussi confirme l’accès au service. La réception d’au moins un événement valide confirme aussi son contenu; si aucun événement n’arrive dans la fenêtre, la réception reste à confirmer lors d’un nouvel essai.

In [5]:
print("--- Bilan ---")
print(f"Connexion : {'réussie' if websocket is not None else 'échouée'}")
print(f"Trames reçues : {len(raw_messages)}")
print(f"JSON décodés : {len(decoded_messages)}")
print(f"Événements valides : {len(valid_events)}")
print(f"Erreurs JSON : {len(decode_errors)}")
print(f"Erreurs de structure : {len(structure_errors)}")

if websocket is None:
    print("Résultat : impossible de joindre le WebSocket EMSC.")
elif not raw_messages:
    print("Résultat : handshake réussi, mais aucun événement reçu dans la fenêtre d’attente.")
elif valid_events:
    print("Résultat : connexion et format d’au moins un événement vérifiés.")
else:
    print("Résultat : connexion établie, mais aucun événement n’a passé la validation.")

if websocket is not None:
    try:
        await websocket.close()
    except Exception as error:
        print(f"Erreur à la fermeture : {type(error).__name__}: {error}")
    else:
        print(f"Connexion WebSocket fermée (code={websocket.close_code}).")

--- Bilan ---
Connexion : réussie
Trames reçues : 1
JSON décodés : 1
Événements valides : 1
Erreurs JSON : 0
Erreurs de structure : 0
Résultat : connexion et format d’au moins un événement vérifiés.
Connexion WebSocket fermée (code=1000).
